# Проверка preprocessing

- Преобразование признаков
- Нормализация train-статистиками
- Оконная выборка 32 → 8


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mden_battery.data import (
    SlidingWindowDataset,
    WindowConfig,
    read_prepared_frame,
)


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "prepared_log_age"
INDEX_PATH = DATA_ROOT / "prepared_index.csv"
SCALER_PATH = DATA_ROOT / "scaler.csv"

INPUT_COLUMNS = [
    "cycle",
    "time_s",
    "voltage_V",
    "current_A",
    "temperature_C",
]
WINDOW_CONFIG = WindowConfig(
    input_length=32,
    horizon=8,
    stride=8,
)
MINIMUM_ROWS = 40


In [ ]:
def load_train_part(
    index_path: Path,
    minimum_rows: int,
) -> pd.DataFrame:
    """Load one train part containing at least one full window."""
    index = pd.read_csv(index_path)
    train_rows = index[index["split"] == "train"]
    for path in train_rows["path"]:
        frame = read_prepared_frame(path)
        if len(frame) >= minimum_rows:
            return frame.reset_index(drop=True)
    raise ValueError("No train part contains a complete window")


In [ ]:
def normalize_inputs(
    frame: pd.DataFrame,
    scaler_path: Path,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Normalize model inputs with persisted train statistics."""
    scaler = pd.read_csv(scaler_path).set_index("feature")
    missing = set(INPUT_COLUMNS) - set(scaler.index)
    if missing:
        raise KeyError(f"Scaler misses features: {sorted(missing)}")

    mean = scaler.loc[INPUT_COLUMNS, "mean"].to_numpy(np.float32)
    std = scaler.loc[INPUT_COLUMNS, "std"].to_numpy(np.float32)
    if not np.all(np.isfinite(std)) or np.any(std <= 0):
        raise ValueError("Scaler contains invalid standard deviations")

    raw = frame[INPUT_COLUMNS].to_numpy(np.float32, copy=True)
    normalized = (raw - mean) / std
    return raw, normalized, mean, std


## Проверки


In [ ]:
assert INDEX_PATH.is_file(), INDEX_PATH
assert SCALER_PATH.is_file(), SCALER_PATH

frame = load_train_part(INDEX_PATH, MINIMUM_ROWS)
required = set(INPUT_COLUMNS) | {"soc", "soh"}
assert required <= set(frame.columns)
assert not frame[list(required)].isna().any().any()

raw, normalized, feature_mean, feature_std = normalize_inputs(
    frame,
    SCALER_PATH,
)
restored = normalized * feature_std + feature_mean

assert np.isfinite(raw).all()
assert np.isfinite(normalized).all()
assert np.allclose(restored, raw, rtol=1e-5, atol=1e-5)


In [ ]:
dataset = SlidingWindowDataset(
    frame,
    INPUT_COLUMNS,
    group_col=None,
    config=WINDOW_CONFIG,
    mean=feature_mean,
    std=feature_std,
)
sample = dataset[0]

assert sample["x"].shape == (32, 5)
assert sample["y_soc"].shape == (8,)
assert sample["y_soh"].shape == (8,)
assert np.allclose(
    sample["x"].numpy(),
    normalized[:32],
    rtol=1e-5,
    atol=1e-5,
)

summary = pd.DataFrame(
    {
        "feature": INPUT_COLUMNS,
        "raw_mean": raw.mean(axis=0),
        "normalized_mean": normalized.mean(axis=0),
        "normalized_std": normalized.std(axis=0),
    }
)
print(summary.to_string(index=False))


## Распределения


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(raw, showfliers=False)
axes[0].set_title("До нормализации")
axes[1].boxplot(
    normalized,
    showfliers=False,
)
axes[1].set_title("После нормализации")
for axis in axes:
    axis.set_xticks(
        range(1, len(INPUT_COLUMNS) + 1),
        INPUT_COLUMNS,
        rotation=30,
    )
    axis.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
